<a href="https://colab.research.google.com/github/Gustavo-kohler/tcc_pinn_fwi/blob/main/pinn_fwi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto de TCC: PINNs para FWI
**Autor:** Gustavo | **Dataset:** OpenFWI (FlatVel-A)

Este notebook configura o ambiente para o treinamento de Redes Neurais Informadas pela Física (PINNs).
O backend numérico utilizado é o **PyTorch**, com abstração de grafos computacionais e geometria gerenciada pelo **DeepXDE**.

### Integração com o Google Drive
Para evitar o carregamento local dos dados do FlatVel-A a cada sessão, montamos o Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Estruturação de Diretórios e Dependências
Aqui criamos a arquitetura de pastas padronizada para projetos de Machine Learning.
Em seguida, instalamos o DeepXDE e outras dependências.

In [2]:
%cd /content/drive/MyDrive/

!mkdir -p tcc_pinn_fwi/data/raw
!mkdir -p tcc_pinn_fwi/src
!mkdir -p tcc_pinn_fwi/models

%cd tcc_pinn_fwi/

!pip install deepxde numpy matplotlib scipy

/content/drive/MyDrive
/content/drive/MyDrive/tcc_pinn_fwi
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 13.0 MB/s eta 0:00:00


### Validação do Motor Matemático e Hardware
Imporação das bibliotécas e configuração do DeepXDE para adotar o PyTorch como backend e validamos a alocação da placa de vídeo (GPU) fornecida pela nuvem.

In [3]:
import os
import torch
import numpy as np

os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde

print("--- DIAGNÓSTICO DO SISTEMA ---")
print(f"Backend do DeepXDE configurado para: {dde.backend.backend_name}")
print(f"Versão do PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"Acelerador de Hardware ATIVO: {torch.cuda.get_device_name(0)}")
    print(f"Memória Total da GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("ALERTA: O ambiente está rodando apenas em CPU. O treinamento da PINN será extremamente lento. Ative a GPU no menu do Colab.")

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


--- DIAGNÓSTICO DO SISTEMA ---
Backend do DeepXDE configurado para: pytorch
Versão do PyTorch: 2.11.0+cu128
Acelerador de Hardware ATIVO: Tesla T4
Memória Total da GPU: 15.64 GB


### Pré-processamento dos Dados
Nesta etapa, isolamos o sismograma de uma única fonte emissora e utilizamos a biblioteca NumPy para mapear o espaço `(1000 tempos x 70 sensores)` em malhas de coordenadas geográficas (`np.meshgrid`).
Em seguida, "achatamos" essa malha (`reshape`) para gerar o conjunto de treinamento tabular:
* Entradas (`X`): Colunas unificadas das posições de cada medição `[x, z, t]`.
* Saídas (`y`): Coluna contendo o valor da pressão acústica medida `[u]`.

In [4]:
data_path = '/content/drive/MyDrive/tcc_pinn_fwi/data/raw/data1.npy'

def load_sensor_data(data_path, id_sample=0, id_source=2):
  data = np.load(data_path)

  seismogram = data[id_sample, id_source, :, :] * 1000.0

  t_array = np.linspace(0.0, 1.0, 1000)
  x_array = np.linspace(0.0, 700.0, 70)

  T_grid, X_grid = np.meshgrid(t_array,x_array, indexing='ij')
  Z_grid = np.zeros_like(X_grid)

  X_input = np.hstack((
      X_grid.reshape(-1, 1),
      Z_grid.reshape(-1, 1),
      T_grid.reshape(-1, 1),
  ))
  y_output = seismogram.reshape(-1, 1)

  return X_input, y_output

X_train, y_train = load_sensor_data(data_path)
print("Formato X (Sensores):", X_train.shape)
print("Formato Y (Pressão):", y_train.shape)


Formato X (Sensores): (70000, 3)
Formato Y (Pressão): (70000, 1)


### Modelagem da Física: A Equação da Onda e o Termo de Fonte

Nesta secção, definimos o resíduo da Equação da Onda Acústica 2D, que dita o comportamento físico da Inversão Sísmica (FWI).

A equação diferencial parcial adotada para a modelação acústica em meios isotrópicos e de densidade constante é:

$$\left( \frac{\partial^2 p}{\partial x^2} + \frac{\partial^2 p}{\partial z^2} \right) - \frac{1}{c^2(x,z)} \frac{\partial^2 p}{\partial t^2} = s(x, z, t)$$

Onde:
* $p$ é o campo de pressão acústica.
* $c(x,z)$ é o modelo de velocidade de propagação do subsolo.
* $s(x, z, t)$ é o termo de injeção da fonte sísmica.

#### Formulação do Termo de Fonte ($s$)
Para que a PINN aproxime a perturbação real sem sofrer instabilidade de gradiente devido a singularidades (como a função Delta de Dirac), o termo de fonte é aproximado por separação de variáveis:

$$s(x, z, t) = R(t) \cdot G(x, z)$$

**1. Domínio do Tempo (Ricker Wavelet):**
A assinatura temporal do pulso é a segunda derivada de uma função Gaussiana. A frequência dominante $f_0$ é fixada em $15$ Hz, conforme a configuração oficial da família *Vel* do OpenFWI.
$$R(t) = \left( 1 - 2\pi^2 f_0^2 (t - t_0)^2 \right) e^{-\pi^2 f_0^2 (t - t_0)^2}$$
* **O papel do atraso temporal ($t_0$):** A simulação inicia em um estado de repouso absoluto ($t=0$). Se o pulso de Ricker não for deslocado no tempo ($t_0 = 0$), o seu pico de energia máxima ocorrerá exatamente no instante inicial, implicando que metade da onda existiria em tempos negativos. Ao aplicar um atraso temporal $t_0$ (ex: $0.1$ s), a curva inteira é deslocada para o futuro, garantindo que a energia cresce suavemente a partir do zero.

**2. Domínio do Espaço (Aproximação Gaussiana):**
A fonte pontual localizada em $(x_s, z_s)$ é suavizada por uma curva de sino para permitir a diferenciação contínua da rede neural.
$$G(x, z) = e^{-\frac{(x - x_s)^2 + (z - z_s)^2}{\sigma^2}}$$
Onde $\sigma$ é um escalar que controla a dispersão espacial da energia ao redor do epicentro.
* **O papel da dispersão espacial ($\sigma$):** Se for excessivamente pequeno, a curva se aproxima de uma singularidade, inviabilizando a otimização da PINN via Diferenciação Automática devido a derivadas extremas. Se for excessivamente grande, o disparo perde a sua natureza pontual e passa a simular uma onda plana, o que distorce a frente de onda esférica esperada nos sismogramas.

**Referências:**
1. Deng, C. et al. (2022). *OPENFWI: Large-scale Multi-structural Benchmark Datasets for Full Waveform Inversion*.
2. Virieux, J., & Operto, S. (2009). *An overview of full-waveform inversion in exploration geophysics*.
3. Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations*.
4. Moseley, B., Markham, A., & Nissen-Meyer, T. (2020). *Solving the wave equation with physics-informed deep learning*.


In [6]:
def pde_wave_acoustic(X, y):
  x_pos = X[:, 0:1]
  z_pos = X[:, 1:2]
  t_time = X[:, 2:3]

  dp_xx = dde.grad.hessian(y, X, i=0, j=0)
  dp_zz = dde.grad.hessian(y, X, i=1, j=1)
  dp_tt = dde.grad.hessian(y, X, i=2, j=2)

  f0 = 15.0       # Frequência dominante (Hz)
  t0 = 0.1        # Atraso do pico (segundos)
  x_fonte = 350.0 # O canhão está no meio do eixo X (350m)
  z_fonte = 0.0   # O canhão está na superfície (0m)
  sigma = 10.0    # Raio de espalhamento da fonte
  pi = np.pi

  ricker_tempo = (1 - 2*torch.square(pi*f0*(t_time-t0))) * torch.exp(-torch.square(pi*f0*(t_time-t0)))

  gaussiana_espaco = torch.exp(-(torch.square(x_pos-x_fonte) + torch.square(z_pos-z_fonte)) / (sigma**2))

  source = ricker_tempo * gaussiana_espaco

  c = 3000.0 # Valor arbitrario temporario.

  residuo = (dp_xx + dp_zz) - (1/(c**2))*dp_tt - source

  return residuo